# MedSched

In [1]:
import numpy as np
import pandas as pd
import random
import jdatetime
from irholiday import irHoliday
import pyprog

In [2]:
interns = {'Mohammad Mahdi Hosseini':0,'Mostafa Hosseini':1,'Sara Hosseini':2,'Mahdi Ramezani':3,
           'Farzin Alizadeh':4,'Masoud Rashnavad':5,'Fatemeh Mohammadi':6,'Sara Daniali':7,
           'Masoome Shabani':8,'Parsa Zadsar':9,'Atephe Charkazi':10,'Yeganeh Noori':11,
           'Fatemeh Golchoob':12, 'Mohammad Sahneh':13, 'Alireza Ramezani':14, 'Amir Alizadeh':15} # dict of interns
couples = [{1,2}, {3,4}, {11,7}]
uncouples = [{8,12},{6,9}]
n_interns = len(interns)
hospitals = {'Rouhani_1':2,'Rouhani_2':2,'Beheshti':1,'Yahya-nejad':1} # dict of hospitals:hardship level
n_hospitals = len(hospitals)
n_date = 31

In [7]:
l = np.arange(n_interns)
t = []
for i in range(n_date*n_hospitals//n_interns + 1):
    random.shuffle(l)
    t.extend(l)
t = np.array(t[:n_date*n_hospitals]).reshape((n_date,n_hospitals)) 
#t = t[:5,]

In [4]:
calendar = irHoliday()
holidays = [jdatetime.date(1402, 2, i).weekday() == 6 for i in range(1,32)]
df = calendar.to_df(1402)
temp = df[df.holiday == 1]
temp = temp[temp.jalali_date >= jdatetime.date(1402,2,1)]
temp = temp[temp.jalali_date <= jdatetime.date(1402,2,31)]
temp = [i.day for i in temp.jalali_date]
for i in temp:
    holidays[i-1] = True
date = []
for idx, i in enumerate(holidays):
    if i: date.append(f'{jdatetime.date(1402, 2, idx+1).strftime("%Y/%m/%d-%a")}-Holi')
    else: date.append(f'{jdatetime.date(1402, 2, idx+1).strftime("%Y/%m/%d-%a")}')
post_keshik = []
for idx, i in enumerate(holidays):
    if i and not holidays[idx+1]:
        post_keshik.append(idx)
##################################################################
def show_table(t, save=False):
    res = np.array([list(interns.keys())[i] for i in t.reshape((1,-1))[0]])
    df = pd.DataFrame(res.reshape((31,-1)),index=date,columns=hospitals.keys())
    if save: 
        df.to_csv("Standby_Table.csv")
        print("Standby-Table is created.")
    return df
##################################################################
def summary(t):
    total = np.unique(t,return_counts=True)
    rouhani = np.unique(t[:,:2],return_counts=True) # rouhani 
    beheshti = np.unique(t[:,2:],return_counts=True) # beheshti yahya
    postkeshik = np.unique(t[post_keshik], return_counts=True) # postkeshik
    df = pd.DataFrame(data = {"total":total[1],
                          "rouhani":rouhani[1],
                          "beheshti":beheshti[1],
                          "postkeshik":postkeshik[1]}, index=interns.keys())
    return df
##################################################################
def nunique(a, axis):
    return (np.diff(np.sort(a,axis=axis),axis=axis)!=0).sum(axis=axis)+1
##################################################################
def calculate_loss(t,report=False):
    l = 0
    if report: print(f'First of all: {l}')
    # specific day for each person
    farzin = list(range(1,32,7))
    shabani = [13,14]
    mostafa = [0,1]
    masoud = [0,1,2,14]
    sahneh = [0,1,2]
    charkazi = [0,1,2]
    yeganeh = [0,1]
    smmh = [0,2]
    
    l += np.sum(t[farzin] == 4)*200
    l += np.sum(t[shabani] == 8)*200
    l += np.sum(t[mostafa] == 1)*200
    l += np.sum(t[masoud] == 5)*200
    l += np.sum(t[sahneh] == 13)*200
    l += np.sum(t[charkazi] == 10)*200
    l += np.sum(t[yeganeh] == 11)*200
    l -= np.sum(t[smmh] == 0)*200

    if report: print(f'After specific day for each person loss: {l}')
    # fair postkeshik 
    t_uniq = np.unique(t,return_counts=True)
    temp_uniq = np.unique(t[post_keshik],return_counts=True)
    l += sum(temp_uniq[1]*temp_uniq[1]*1.5**t_uniq[1][temp_uniq[0]])
    if report: print(f'For Post_keshik loss: {l}')
    # Hospital fair devision 
    for i in np.unique(list(hospitals.values())):
        temp = t[:,np.array(list(hospitals.values())) == i]
        temp_uniq = np.unique(temp,return_counts=True)
        l += sum(temp_uniq[1]*temp_uniq[1]*i*1.5**t_uniq[1][temp_uniq[0]])
    
    if report: print(f'After Hospital fair division: {l}')
    # Two standby in one day 
    l += (t.shape[1]*t.shape[0] - nunique(t,1).sum())*1000
    if report: print(f'After two standby of a person in one day: {l}')
    # Two standby in a row
    for i in range(t.shape[0]-1):
        l += len(np.intersect1d(t[i],t[i+1]))*500
    if report: print(f'After two standby in a row: {l}')
    # Couple in one day 
    for i in range(t.shape[0]):
        for j in couples:
            if set(t[i,:2]) == j:
                l -= 100
    if report: print(f'After couple loss: {l}')
    # uncouple in different day 
    for i in range(t.shape[0]):
        for j in uncouples:
            if set(t[i,:2]) == j:
                l += 100
    if report: print(f'After uncouple loss: {l}')
    return int(l)
###################################################################
def transfer(t,idx,jdx): #  
    score = np.zeros_like(t,dtype=int)+calculate_loss(t)
    t1 = t.copy()
    for i in range(len(t)):
        for j in range(t.shape[1]):
            t1 = t.copy()
            t1[idx,jdx],t1[i,j] = t1[i,j],t1[idx,jdx]
            score[i,j] = calculate_loss(t1)
    return score
###################################################################

def iteration(epoch,t):
    for e in range(epoch):
        print(f"epoch: {e+1}, loss = {calculate_loss(t)}")
        prog = pyprog.ProgressBar(" ", "", 31)
        prog.update()
        for i in range(t.shape[0]):
            prog.set_stat(i + 1)
            prog.update()
            for j in range(t.shape[1]):
                s = transfer(t,i,j)
                x = np.where(s==s.min())
                x = list(zip(x[0], x[1]))
                random.shuffle(x)
                if s.min() <= calculate_loss(t):
                    t[x[0][0],x[0][1]], t[i,j] = t[i,j], t[x[0][0],x[0][1]]
        prog.end()

In [8]:
iteration(6,t)

epoch: 1, loss = 23512
 Progress: 100% ################################################## 
epoch: 2, loss = 16996
 Progress: 100% ################################################## 
epoch: 3, loss = 16796
 Progress: 100% ################################################## 
epoch: 4, loss = 16722
 Progress: 100% ################################################## 
epoch: 5, loss = 16596
 Progress: 100% ################################################## 
epoch: 6, loss = 16596
 Progress: 100% ################################################## 


In [9]:
show_table(t)

,Rouhani_1,Rouhani_2,Beheshti,Yahya-nejad
1402/02/01-Fri-Holi,Amir Alizadeh,Fatemeh Golchoob,Mohammad Mahdi Hosseini,Masoome Shabani
1402/02/02-Sat-Holi,Parsa Zadsar,Alireza Ramezani,Mahdi Ramezani,Sara Hosseini
1402/02/03-Sun-Holi,Yeganeh Noori,Sara Daniali,Mohammad Mahdi Hosseini,Fatemeh Golchoob
1402/02/04-Mon,Masoud Rashnavad,Mohammad Sahneh,Fatemeh Mohammadi,Amir Alizadeh
1402/02/05-Tue,Mostafa Hosseini,Sara Hosseini,Mahdi Ramezani,Parsa Zadsar
1402/02/06-Wed,Yeganeh Noori,Sara Daniali,Amir Alizadeh,Mohammad Mahdi Hosseini
1402/02/07-Thu,Masoud Rashnavad,Masoome Shabani,Mohammad Sahneh,Alireza Ramezani
1402/02/08-Fri-Holi,Farzin Alizadeh,Mahdi Ramezani,Parsa Zadsar,Mostafa Hosseini
1402/02/09-Sat,Yeganeh Noori,Sara Daniali,Mohammad Sahneh,Fatemeh Mohammadi
1402/02/10-Sun,Mostafa Hosseini,Sara Hosseini,Amir Alizadeh,Masoud Rashnavad


In [10]:
summary(t)

,total,rouhani,beheshti,postkeshik
Mohammad Mahdi Hosseini,8,4,4,2
Mostafa Hosseini,8,4,4,1
Sara Hosseini,8,4,4,1
Mahdi Ramezani,8,4,4,1
Farzin Alizadeh,8,4,4,2
Masoud Rashnavad,7,4,3,2
Fatemeh Mohammadi,8,3,5,1
Sara Daniali,7,4,3,2
Masoome Shabani,7,4,3,2
Parsa Zadsar,7,4,3,2
